In [ ]:
import pandas as pd
import numpy as np
import gzip
from io import StringIO
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, MultiHeadAttention, LayerNormalization, add, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.metrics import Precision, Recall


# read data and preprocessing
def read_data(filename):
    threshold = 1800
    max_tokens = 27

    def pad_sequence(sequence):
        return sequence + "-" * (threshold - len(sequence))

    file_path = filename + '.tsv.gz'
    with gzip.open(file_path, 'rt') as f:
        content = f.read()

    df = pd.read_csv(StringIO(content), sep='\t')
    df = df.dropna(subset=['Gene Ontology (GO)', 'Sequence'])
    df = df[~df["Sequence"].duplicated()]
    df = df[df['Sequence'].apply(lambda x: len(x)) <= threshold]
    df.reset_index(drop=True, inplace=True)
    df['Sequence'] = df['Sequence'].apply(pad_sequence)

    char_vals = {"M":1,"N":2,"S":3,"K":4,"I":5,"F":6,"A":7,"V":8,"L":9,"G":10,
                 "R":11,"C":12,"D":13,"Q":14,"Y":15,"P":16,"T":17,"E":18,
                 "H":19,"W":20,"X":21,"U":22,"Z":23,"B":24,"O":25,"-":26}
    df['Sequence'] = df['Sequence'].apply(lambda x: [char_vals[char] for char in list(x)])

    return df


def build_transformer_model(input_shape, max_tokens, embed_dim, num_heads):
    inputs = Input(shape=input_shape)
    embed = Embedding(input_dim=max_tokens, output_dim=embed_dim)(inputs)

    x = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(embed, embed)
    x = add([x, embed])
    x = LayerNormalization()(x)
    residual = x

    x = Dense(embed_dim, activation="relu")(x)
    x = add([x, embed, residual])
    x = GlobalAveragePooling1D()(x)

    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)

    return model


def build_lstm_model(input_shape, max_tokens, embed_dim, lstm_units):
    inputs = Input(shape=input_shape)
    embed = Embedding(input_dim=max_tokens, output_dim=embed_dim)(inputs)

    x = LSTM(units=lstm_units)(embed)

    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)

    return model

# read and build dataset
dev_df_positive = read_data("Extracellular_matrix_organization")
dev_df_negative = read_data("Not_extracellular_matrix_organization")

dev_df_positive['target'] = 1
dev_df_negative['target'] = 0

dev_df = pd.concat([dev_df_positive, dev_df_negative])
dev_df = dev_df.sample(frac=1).reset_index(drop=True)

# split test and train dataset
X_dev, X_test, y_dev, y_test = train_test_split(dev_df['Sequence'], dev_df['target'], test_size=0.2)

X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.333)
# Roughly 10,000 for test and validation

X_train = np.array(X_train.tolist())
X_val = np.array(X_val.tolist())
X_test = np.array(X_test.tolist())

In [ ]:

# build model
transformer_model = build_transformer_model(input_shape=(1800,), max_tokens=27, embed_dim=200, num_heads=4)
transformer_model.compile(optimizer=RMSprop(learning_rate=0.01), loss="binary_crossentropy", metrics=["acc", Precision(), Recall()])
# train multiple times for eliminate bias
transformer_model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_val, y_val))



1444/1444 [==============================] - 879s 607ms/step - loss: 0.4738 - acc: 0.8023 - precision: 0.8305 - recall: 0.7210 - val_loss: 0.3895 - val_acc: 0.8376 - val_precision: 0.8427 - val_recall: 0.8012


In [ ]:
transformer_model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_val, y_val))

1444/1444 [==============================] - 882s 611ms/step - loss: 0.3719 - acc: 0.8419 - precision: 0.8645 - recall: 0.7819 - val_loss: 0.3446 - val_acc: 0.8592 - val_precision: 0.8783 - val_recall: 0.8103


In [ ]:
transformer_model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_val, y_val))

1444/1444 [==============================] - 881s 610ms/step - loss: 0.3475 - acc: 0.8562 - precision: 0.8818 - recall: 0.7969 - val_loss: 0.3433 - val_acc: 0.8624 - val_precision: 0.9158 - val_recall: 0.7762


In [ ]:
transformer_model.fit(X_train, y_train, epochs=1, batch_size=32, validation_data=(X_val, y_val))

1444/1444 [==============================] - 881s 610ms/step - loss: 0.3390 - acc: 0.8630 - precision: 0.8897 - recall: 0.8047 - val_loss: 0.3463 - val_acc: 0.8631 - val_precision: 0.8646 - val_recall: 0.8375


In [ ]:
transformer_model.fit(X_val, y_val, epochs=1, batch_size=32)

721/721 [==============================] - 373s 517ms/step - loss: 0.3433 - acc: 0.8586 - precision: 0.8869 - recall: 0.7985


In [ ]:
transformer_model.fit(X_test, y_test, epochs=1, batch_size=32, validation_split=.9999)

1/1 [==============================] - 100s 100s/step - loss: 0.0024 - acc: 1.0000 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.3688 - val_acc: 0.8572 - val_precision: 0.9583 - val_recall: 0.7209


In [ ]:
len(X_test)

17317

In [ ]:
# build model
lstm_model = build_lstm_model(input_shape=(1800,), max_tokens=27, embed_dim=100, lstm_units=64)
lstm_model.compile(optimizer=RMSprop(learning_rate=0.01), loss="binary_crossentropy", metrics=["acc", Precision(), Recall()])
# train multiple times
lstm_model.fit(X_train, y_train, epochs=1, batch_size=64, validation_data=(X_val, y_val))

722/722 [==============================] - 47s 62ms/step - loss: 0.6913 - acc: 0.5337 - precision_1: 0.4531 - recall_1: 0.0230 - val_loss: 0.6909 - val_acc: 0.5339 - val_precision_1: 0.0000e+00 - val_recall_1: 0.0000e+00


In [ ]:
lstm_model.fit(X_train, y_train, epochs=1, batch_size=64, validation_data=(X_val, y_val))

722/722 [==============================] - 44s 61ms/step - loss: 0.6909 - acc: 0.5358 - precision_1: 0.4948 - recall_1: 0.0089 - val_loss: 0.6908 - val_acc: 0.5339 - val_precision_1: 0.0000e+00 - val_recall_1: 0.0000e+00


In [ ]:
lstm_model.fit(X_test, y_test, epochs=1, batch_size=32, validation_split=.9999)

1/1 [==============================] - 11s 11s/step - loss: 0.6090 - acc: 1.0000 - precision_1: 0.0000e+00 - recall_1: 0.0000e+00 - val_loss: 0.6909 - val_acc: 0.5401 - val_precision_1: 0.0000e+00 - val_recall_1: 0.0000e+00


In [ ]:
transformer_model.save('transformer_ecm_model.keras')
lstm_model.save('lstm_ecm_model.keras')